In [43]:
import pandas as pd
from dotenv import dotenv_values
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions
import os
from groq import Groq
from dotenv import dotenv_values
from pprint import pprint
from google import genai
from google.genai import types

In [44]:
# Load Env Varaibles + Import Function Help Us to Embeding Documents
config = dotenv_values(".env")
GROQ_API_KEY = config["GROQ_API_KEY"]
GEMENI_API_KEY = config["GEMENI_API_KEY"]

In [45]:
# try Gemeni Embedding Function
gemeni_client = genai.Client(api_key=GEMENI_API_KEY)
result = gemeni_client.models.embed_content(
        model="gemini-embedding-2",
        contents="What is the meaning of life?"
)
print(result.embeddings[0].values)

[-0.016332133, -0.0043764366, -0.0011324773, -0.011240026, 0.00029597108, 0.0014934157, -0.018807797, 0.013529229, 0.019723475, -0.049145423, -0.020023376, 0.019897161, -0.025475917, -0.016769981, 0.011656811, 0.014576932, 0.017489318, 0.0019741745, -0.033542495, -0.0052898847, -0.01738165, -0.0071869395, 0.005816727, 0.0036677625, -0.004281101, 0.010329708, -0.005853738, 0.007902185, -0.0031149625, 0.13209803, -0.016384594, 0.010829647, -0.00854677, 0.0034034406, 0.0017756857, 0.015020709, 0.00256653, -0.00677215, 0.014306844, -0.010287376, 0.009519303, 0.010457309, 0.009206434, 0.012812066, -0.010743783, -0.00047993741, -0.020863462, -0.005375269, 0.0116195325, 0.00081031286, -0.006415543, 0.024154454, -0.008800871, -0.039657805, -0.0070156506, -0.013774676, 0.0153811, -0.0025023571, 0.013820424, 0.018328026, -0.025549209, 0.0067770947, 0.0013363153, 0.029920602, -0.019204818, -0.007714158, 0.020704817, -0.029377379, 0.026432242, -0.0097309025, 0.004420867, 0.0077373628, -0.025789557

In [46]:
os.environ["GEMINI_API_KEY"] = GEMENI_API_KEY
google_ef = embedding_functions.GoogleGeminiEmbeddingFunction(
    model_name="gemini-embedding-001",
    task_type="RETRIEVAL_DOCUMENT",
    dimension=768,
)

In [47]:
# Init Chroma Vector Database
chroma_client = chromadb.PersistentClient(path="articles")
collecation_name = "articles"
collection = chroma_client.get_or_create_collection(name=collecation_name, embedding_function=google_ef)
print(collection)

Collection(name=articles)


In [48]:
# Function to load documents from a directory
def load_documents_from_directory(directory_path):
    documents = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".txt"):
            with open(os.path.join(directory_path, filename), "r", encoding="utf-8") as file:
                documents.append({"id": filename, "text": file.read()})
    return documents

In [49]:
# Function to split text into chunks
def split_text(text, chunk_size=1000, chunk_overlap=20):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - chunk_overlap
    return chunks

In [50]:
# Load documents from the directory
directory_path = "./news_articles/"
documents = load_documents_from_directory(directory_path=directory_path)
print(f"Number Of Documents: {len(documents)}")

Number Of Documents: 21


In [51]:
# Split documents into chunks
chunked_documents = []
for doc in documents:
    chunks = split_text(text=doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_documents.append({"id": f"{doc["id"]}_chunk{i+1}", "text": chunk})
print(chunked_documents)

[{'id': '05-06-amazon-launches-free-channels-check-marks-come-to-gmail-and-openai-raises-more-moolah.txt_chunk1', 'text': 'It’s that time of week again, folks — Week in Review (WiR) time. For those new to the scene, WiR is TechCrunch’s regular newsletter that recaps the biggest tech stories over the past few days. There’s no better digest for the person on the go, we’d argue — but of course, we’re a little biased.\n\nBefore we get into the meat of the thing, a quick reminder that TC City Spotlight: Atlanta is fast approaching. On June 7, TechCrunch is headed to Atlanta, where we’ll host a pitch competition, a talk on the economics of equality, a panel discussion on investing in the Atlanta ecosystem and more.\n\nElsewhere, there’s a TechCrunch Live event with Persona and Index Ventures on May 10, which will touch on how Persona keeps pace with new threats and how Index made a prescient move to spot and back Persona early on. And we have Disrupt in San Francisco from September 19–21 — o

In [52]:
# Function to Generate embeddings for the document chunks - Doc2Vect
def get_gemeni_embedding(text):
    response = gemeni_client.models.embed_content(
        model="gemini-embedding-2",
        contents=text,
        config=types.EmbedContentConfig(
            output_dimensionality=768
        )
)
    embedding = response.embeddings[0].values
    return embedding

In [54]:
# Get Embedding Vectors
chunked_documents = chunked_documents[0:3]
for doc_chunk in chunked_documents:
    doc_chunk['embeddings'] = get_gemeni_embedding(doc_chunk["text"])
    print(len(doc_chunk['embeddings']))


768
768
768


In [55]:
for doc_chunk in chunked_documents:
    print(doc_chunk["embeddings"])

[-0.02829606, 0.0015243453, 0.04944331, -0.005880625, 0.045167517, 0.04722697, -0.01721351, 0.036983732, 0.08607504, -0.0820949, -0.018423095, 0.0055500674, -0.018479833, -0.03297445, -0.019136827, -0.0052461745, 0.0056921197, -0.0043030754, 0.006525761, -0.023939725, -0.024854075, 0.055517923, -0.015940271, -0.0022185002, -0.046482183, 0.011509714, 0.0035869265, -0.0027625964, -0.019440409, 0.13189322, -0.07230879, -0.023936711, 0.042556595, 0.036058545, 0.030468816, -0.0026451114, 0.027152508, -0.023527538, -0.013637676, -0.025211873, 0.017214244, 0.042002372, -0.005924083, -0.028742464, -0.0035213786, -0.015399911, -0.014376878, -0.013297884, -0.023030985, 0.025769036, -0.052616205, -0.018128898, -0.012623106, 0.028652886, 0.014069693, -0.013455838, 0.039650165, -0.013077341, -0.041018043, -0.022269024, -0.010762095, -0.0050745695, -0.02239753, 0.07603175, 0.0020012045, -0.056017697, 0.03189403, 0.032688156, -0.0034681961, 0.028314572, 0.03747578, -0.021684056, -0.045304902, -0.0244

In [56]:
# Save Embedding Into Vector Database
for doc_chunk in chunked_documents:
    collection.upsert(ids=[doc_chunk["id"]], documents=[doc_chunk["text"]], embeddings=[doc_chunk["embeddings"]])

collection.get(ids=["05-06-amazon-launches-free-channels-check-marks-come-to-gmail-and-openai-raises-more-moolah.txt_chunk1"],
    include=["embeddings"])

{'ids': ['05-06-amazon-launches-free-channels-check-marks-come-to-gmail-and-openai-raises-more-moolah.txt_chunk1'],
 'embeddings': array([[-2.82960627e-02,  1.52434537e-03,  4.94433120e-02,
         -5.88062545e-03,  4.51675206e-02,  4.72269729e-02,
         -1.72135103e-02,  3.69837321e-02,  8.60750452e-02,
         -8.20949078e-02, -1.84230953e-02,  5.55006787e-03,
         -1.84798334e-02, -3.29744518e-02, -1.91368274e-02,
         -5.24617499e-03,  5.69212018e-03, -4.30307537e-03,
          6.52576098e-03, -2.39397269e-02, -2.48540770e-02,
          5.55179268e-02, -1.59402713e-02, -2.21850025e-03,
         -4.64821868e-02,  1.15097146e-02,  3.58692673e-03,
         -2.76259659e-03, -1.94404088e-02,  1.31893218e-01,
         -7.23087937e-02, -2.39367131e-02,  4.25565988e-02,
          3.60585451e-02,  3.04688178e-02, -2.64511164e-03,
          2.71525085e-02, -2.35275403e-02, -1.36376759e-02,
         -2.52118744e-02,  1.72142442e-02,  4.20023762e-02,
         -5.92408329e-03, -2.8

In [57]:
# Function That Query Documents
'''
    collection.query(): find the documents most related to text
        - query_texts: the question or text you want to search with.
        - n_results: how many documents we want back.
'''
def query_documents(question, n_results=2):
    relevant_chunks = collection.query(
        query_texts=question,
        n_results=n_results
    )
    
    # Extract The Relevent Chunks
    relevant_chunks = [doc for sublist in relevant_chunks["documents"] for doc in sublist]
    return relevant_chunks

query_documents("What is machine learning?", n_results=100)

['ogle Account users globally, roughly a year after the company — alongside Apple, Microsoft and the FIDO Alliance — announced a partnership to broadly advance passkey adoption. With passkeys, users’ authentication synchronizes across devices through the cloud using cryptographic key pairs, allowing them to sign in to websites and apps using the same biometrics or screen-lock PIN they use to unlock their devices.\n\nMicrosoft debuts Pegasus: Microsoft this week announced that it’ll extend the Startup Founders Hub, its self-service platform that provides founders with free resources, including Azure credits, with a new incubator program called the Pegasus Program. Pegasus will select startups with products that “fill a market need” and give them up to $350,000 in Azure, GitHub and LinkedIn credits plus backing from advisors, as well as “access to the best Microsoft tech,” Microsoft says.\n\nBlue check marks come to Gmail: Google is going to start displaying a blue check mark next to sel

In [89]:
client_groq = Groq(api_key=GROQ_API_KEY)
completion = client_groq.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[
    {
        "role": "system",
        "content": "What Is Machine Learning ? You are an assistant for question-answering tasks. Use the following pieces of "
        "retrieved context to answer the question. If you don't know the answer, say that you " +
        "don't know. Use three sentences maximum and keep the answer concise." + "ogle Account users globally, roughly a year after the company — alongside Apple, Microsoft and the FIDO Alliance — announced a partnership to broadly advance passkey adoption. With passkeys, users authentication synchronizes across devices through the cloud using cryptographic key pairs, allowing them to sign in to websites and apps using the same biometrics or screen-lock PIN they use to unlock their devices.",
    },
    ],
    temperature=1,
    max_completion_tokens=1024,
    top_p=1,
    stream=True,
    stop=None
)

res = []
for chunk in completion:
    res.append(chunk.choices[0].delta.content)

print(type(res))
res = [s for s in res if s]
res = "".join(res)
print(res)

<class 'list'>
I don't know the answer to this question based on the given context about passkeys. However, I can provide a general definition of machine learning: Machine learning is a subset of artificial intelligence that enables computers to learn from data and improve their performance on a task without being explicitly programmed. It involves creating algorithms that can recognize patterns, make predictions, and optimize processes based on experience and input.


In [106]:
# Function That Generate Reponse
client_groq = Groq(api_key=GROQ_API_KEY)
def generate_reponse(question, relevant_chunks):
    context = "".join(relevant_chunks)
    sys_prompt = f"""
        You are an assistant for question-answering tasks. Use the following pieces of 
        retrieved context to answer the question. If you don't know the answer, say that you
        don't know. Use three sentences maximum and keep the answer concise.
            Instructions:
            - Be helpful and answer questions concisely. If you don't know the answer, say 'I don't know'
            - Utilize the context provided for accurate and specific information.
            - Incorporate your preexisting knowledge to enhance the depth and relevance of your response.
            - Cite your sources
        Context :{context}
        """
    completion = client_groq.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
        {
            "role": "system",
            "content": sys_prompt,
        },
        {
            "role": "user",
            "content": question
        }
        ],
        temperature=1,
        max_completion_tokens=1024,
        top_p=1,
        stream=True,
        stop=None
    )

    res = []
    for chunk in completion:
        res.append(chunk.choices[0].delta.content)

    res = [s for s in res if s]
    res = "".join(res)
    return res

question = "What is Machine Learning ?"
relevant_chunks = query_documents(question, n_results=100)
response = generate_reponse(question=question, relevant_chunks=relevant_chunks)
print(response)

I don't know. The provided context does not mention machine learning. However, based on pre-existing knowledge, machine learning is a field of study that focuses on the development of algorithms and statistical models that enable computers to perform a specific task without being explicitly programmed for it.
